In [ ]:
import os
import random
import copy
import numpy as np
import pandas as pd
from PIL import Image, UnidentifiedImageError
from sklearn.model_selection import train_test_split
from tqdm import tqdm

import subprocess
subprocess.run(["pip", "install", "open-clip-torch"], check=True)
import open_clip

import torch
import torch.nn as nn
import torch.optim as optim
from torch.cuda.amp import GradScaler, autocast
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torchvision.transforms as transforms

torch.backends.cudnn.benchmark = True

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

In [ ]:
BASE_INPUT      = "/kaggle/input/competitions/cse-281-spring-26-scene-style-classification"
BASE            = os.path.join(BASE_INPUT, "StyleClassificationIndoors/StyleClassificationIndoors")
SAMPLE_SUB_PATH = os.path.join(BASE_INPUT, "sample_submission.csv")
TRAIN_DIR       = os.path.join(BASE, "train")
TEST_DIR        = os.path.join(BASE, "test")
BATCH_SIZE      = 16

class_names   = sorted(os.listdir(TRAIN_DIR)) #gives foldernames alphabetically
class_mapping = {name: idx for idx, name in enumerate(class_names)}
NUM_CLASSES   = len(class_names)
print(f"Classes ({NUM_CLASSES}):", class_mapping)

all_paths, all_labels = [], []
for class_name, class_id in class_mapping.items():
    class_dir = os.path.join(TRAIN_DIR, class_name)
    if not os.path.isdir(class_dir): continue
    for fname in os.listdir(class_dir):
        if fname.lower().endswith((".jpg", ".jpeg", ".png")):
            all_paths.append(os.path.join(class_dir, fname))
            all_labels.append(class_id)

train_paths, val_paths, train_labels, val_labels = train_test_split(
    all_paths, all_labels, test_size=0.2, random_state=SEED, stratify=all_labels
)

print(f"Train: {len(train_paths)} | Val: {len(val_paths)}")

In [ ]:
GROUP_MAP = {
    'minimalist': 0, 'modern': 0, 'contemporary': 0, 'scandinavian': 0,
    'boho': 1, 'eclectic': 1,
    'mediterranean': 2, 'southwestern': 2, 'french-country': 2,
    'coastal': 3, 'tropical': 3,
    'industrial': 4,
    'farmhouse': 5, 'shabby-chic-style': 5, 'craftsman': 5,   # ← moved back
    'victorian': 6, 'asian': 6,                                # ← craftsman removed
}

GROUP_NAMES = [
    'Modern/Minimal',
    'Global/Maximalist',
    'Sunbaked & Warm Terrains',
    'Bright/Airy',
    'Industrial',
    'Rustic Wood',
    'Ornate/Eastern',
]
NUM_GROUPS = 7


In [ ]:
INPUT_SIZE = 224
EVA_MEAN = [0.48145466, 0.4578275,  0.40821073]
EVA_STD  = [0.26862954, 0.26130258, 0.27577711]

# augmentation
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(INPUT_SIZE, scale=(0.85, 1.0)),  # ← replaces Resize
    transforms.RandAugment(num_ops=2, magnitude=9),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
    transforms.RandomGrayscale(p=0.05),
    transforms.ToTensor(),
    transforms.Normalize(EVA_MEAN, EVA_STD)
])

val_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(INPUT_SIZE),  #added resize and randcrop
    transforms.ToTensor(),
    transforms.Normalize(EVA_MEAN, EVA_STD)
])


class SceneDataset(Dataset):
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths
        self.labels      = labels
        self.transform   = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        try:
            image = Image.open(self.image_paths[idx]).convert("RGB")
            if self.transform:
                image = self.transform(image)
            return image, self.labels[idx]
        except (UnidentifiedImageError, IOError, OSError, TypeError):
            return torch.zeros((3, INPUT_SIZE, INPUT_SIZE)), 0

print("Transforms and Dataset class ready")

In [ ]:
all_paths, all_labels = [], []
for class_name, class_id in class_mapping.items():
    class_dir = os.path.join(TRAIN_DIR, class_name)
    if not os.path.isdir(class_dir): continue
    group_id = GROUP_MAP[class_name]   #  use group, not class_id
    for fname in os.listdir(class_dir):
        if fname.lower().endswith((".jpg", ".jpeg", ".png")):
            all_paths.append(os.path.join(class_dir, fname))
            all_labels.append(group_id)

train_paths, val_paths, train_labels, val_labels = train_test_split(
    all_paths, all_labels, test_size=0.2, random_state=SEED, stratify=all_labels
)

# Weighted sampler to balance groups during training
class_counts   = np.bincount(train_labels)
sample_weights = [1.0 / class_counts[l] for l in train_labels]
sampler        = WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)

train_loader_s1 = DataLoader(
    SceneDataset(train_paths, train_labels, train_transform),
    batch_size=BATCH_SIZE, sampler=sampler,
    num_workers=4, pin_memory=True, persistent_workers=True, prefetch_factor=2,
    drop_last=True
)
val_loader_s1 = DataLoader(
    SceneDataset(val_paths, val_labels, val_transform),
    batch_size=BATCH_SIZE, shuffle=False,
    num_workers=4, pin_memory=True, persistent_workers=True, prefetch_factor=2
)
print(f"Stage 1 data | Train: {len(train_paths)} | Val: {len(val_paths)} | Groups: {NUM_GROUPS}")


WEIGHTS_PATH = "/kaggle/input/datasets/janagohary/evadata/eva_clip_weights.pt"
EMBED_DIM    = 768

def load_eva_backbone():
    """Loads a fresh copy of EVA02 backbone with pretrained weights.
    We call this multiple times — once for Stage 1, once per Stage 2 group.
    Each model gets its own backbone so their weights don't interfere."""
    eva_model, _, _ = open_clip.create_model_and_transforms('EVA02-L-14', pretrained=None)
    checkpoint = torch.load(WEIGHTS_PATH, map_location='cpu', weights_only=False)
    eva_model.load_state_dict(checkpoint, strict=True)
    return eva_model.visual, checkpoint

print("Loading backbone for Stage 1...")
backbone_s1, checkpoint = load_eva_backbone()
print("Backbone ready")

In [ ]:
class EVAClassifier(nn.Module):
    def __init__(self, backbone, embed_dim, num_classes):
        super().__init__()
        self.backbone   = backbone
        self.classifier = nn.Sequential(
            nn.Linear(embed_dim, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        features = self.backbone(x)
        return self.classifier(features)

print("EVAClassifier architecture ready")
# CELL 6.5: Build Stage 1 model
model_stage1 = EVAClassifier(backbone_s1, EMBED_DIM, NUM_GROUPS).to(device)
print(f"Stage 1 model ready — {NUM_GROUPS} group outputs")

In [ ]:
def freeze_backbone_except_last_n_blocks(model, n_unfrozen_blocks=4):
    """Freeze all backbone params except the last N transformer blocks + final norm.
    This reduces overfitting by giving the optimizer far fewer parameters to memorize with."""
    for param in model.backbone.parameters():
        param.requires_grad = False

    blocks = None
    for module in model.backbone.modules():
        if hasattr(module, 'blocks') and isinstance(module.blocks, nn.ModuleList):
            blocks = module.blocks
            break

    if blocks is None:
        print("⚠️ Could not locate transformer blocks automatically — backbone stays fully frozen except classifier.")
        return

    for block in blocks[-n_unfrozen_blocks:]:
        for param in block.parameters():
            param.requires_grad = True

    for module in model.backbone.modules():
        if hasattr(module, 'norm') and isinstance(module.norm, nn.LayerNorm):
            for param in module.norm.parameters():
                param.requires_grad = True

    trainable = sum(p.numel() for p in model.backbone.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.backbone.parameters())
    print(f"Backbone trainable params: {trainable:,} / {total:,} ({trainable/total*100:.1f}%)")

In [ ]:
# CELL 8: Training Function
# - Removed SimilarityAwareLoss (not needed with groups)
# - Uses plain CrossEntropyLoss passed in from outside
# - Updated amp API to remove deprecation warnings

def run_epoch(model, loader, optimizer, criterion, device, scaler, is_train):
    model.train() if is_train else model.eval()
    running_loss = running_correct = total_samples = 0
    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        if is_train:
            optimizer.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast():
                outputs = model(images)
                loss    = criterion(outputs, labels)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            with torch.no_grad(), torch.cuda.amp.autocast():
                outputs = model(images)
                loss    = criterion(outputs, labels)
        running_loss    += loss.item() * images.size(0)
        running_correct += (outputs.argmax(1) == labels).sum().item()
        total_samples   += images.size(0)
    return running_loss / total_samples, running_correct / total_samples


def train_model(model, train_loader, val_loader, num_epochs, backbone_lr, head_lr, label, patience=3):
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    optimizer = optim.Adam([
     {'params': [p for p in model.backbone.parameters() if p.requires_grad], 'lr': backbone_lr},
     {'params': model.classifier.parameters(), 'lr': head_lr},
], weight_decay=2e-3)   # was 1e-3
    scheduler         = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)
    scaler            = GradScaler()
    best_val_acc      = 0.0
    best_wts          = copy.deepcopy(model.state_dict())
    epochs_no_improve = 0

    print(f"\n{'='*50}")
    print(f"Training {label} for up to {num_epochs} epochs (patience={patience})")
    print(f"{'='*50}")

    for epoch in range(num_epochs):
        t_loss, t_acc = run_epoch(model, train_loader, optimizer, criterion, device, scaler, True)
        v_loss, v_acc = run_epoch(model, val_loader,   None,      criterion, device, scaler, False)
        scheduler.step()
        print(f"Epoch {epoch+1}/{num_epochs} | Train: {t_acc*100:.2f}% | Val: {v_acc*100:.2f}%")

        if v_acc > best_val_acc:
            best_val_acc      = v_acc
            best_wts          = copy.deepcopy(model.state_dict())
            epochs_no_improve = 0
            print(f"  ✅ Best saved ({best_val_acc*100:.2f}%)")
        else:
            epochs_no_improve += 1
            print(f"  ⚠️ No improvement ({epochs_no_improve}/{patience})")
            if epochs_no_improve >= patience:
                print(f"  🛑 Early stopping at epoch {epoch+1}")
                break

    model.load_state_dict(best_wts)
    print(f"Done. Best Val Acc: {best_val_acc*100:.2f}%")
    return model


print("Training functions ready")
freeze_backbone_except_last_n_blocks(model_stage1, n_unfrozen_blocks=6)
model_stage1 = train_model(
    model        = model_stage1,
    train_loader = train_loader_s1,
    val_loader   = val_loader_s1,
    num_epochs   = 15,
    backbone_lr  = 3e-6,   # ← lower, was 1e-5
    head_lr      = 5e-5,   # ← lower, was 1e-4
    label        = 'Stage1 (Group Classifier)',
    patience     = 5       # ← more patience, was 3
)

torch.save(model_stage1.state_dict(), "/kaggle/working/model_stage1.pt")
print("Stage 1 model saved")
model_stage1 = model_stage1.cpu()
torch.cuda.empty_cache()

In [ ]:
# CELL: STAGE 1 CONFUSION MATRIX
# Check which groups get confused with which

from sklearn.metrics import confusion_matrix

model_stage1 = model_stage1.to(device)
model_stage1.eval()

all_preds, all_true = [], []

with torch.no_grad():
    for images, labels in tqdm(val_loader_s1):
        images = images.to(device, non_blocking=True)
        with torch.cuda.amp.autocast():
            outputs = model_stage1(images)
        preds = outputs.argmax(1).cpu().numpy()
        all_preds.extend(preds)
        all_true.extend(labels.numpy())

cm = confusion_matrix(all_true, all_preds, labels=list(range(NUM_GROUPS)))

print("Confusion Matrix (rows = true group, cols = predicted group)\n")
header = "        " + "".join(f"{i:>6}" for i in range(NUM_GROUPS))
print(header)
for i, row in enumerate(cm):
    row_str = "".join(f"{v:>6}" for v in row)
    print(f"True {i} | {row_str}   ({GROUP_NAMES[i]})")

print("\nPer-group accuracy:")
for i in range(NUM_GROUPS):
    total = cm[i].sum()
    correct = cm[i][i]
    acc = correct / total * 100 if total > 0 else 0
    print(f"  Group {i} ({GROUP_NAMES[i]}): {acc:.2f}%  ({correct}/{total})")

print("\nTop confused group pairs:")
pairs = []
for i in range(NUM_GROUPS):
    for j in range(NUM_GROUPS):
        if i != j and cm[i][j] > 0:
            pairs.append((cm[i][j], i, j))
pairs.sort(reverse=True)
for count, true_g, pred_g in pairs[:10]:
    print(f"  True={GROUP_NAMES[true_g]:<28} → Predicted={GROUP_NAMES[pred_g]:<28}  ({count} images)")
model_stage1 = model_stage1.cpu()
torch.cuda.empty_cache()

In [ ]:

# CELL 10: BUILD STAGE 2 DATA  ← NEW
#
# For each group, we build a separate dataset
# containing ONLY the images from that group,
# and labels are the original 17-class IDs.
#
# Example for Group 4 (Bold/Dark):
#   asian=0, industrial=8, victorian=16  (original IDs)
#   But locally we remap them to 0, 1, 2
#   so the model head has 3 outputs.
#   We store the mapping so we can convert back later.


def build_stage2_data(group_id):
    """
    Builds train/val data for one Stage 2 model.
    Returns:
        train_paths, val_paths, train_labels, val_labels  (local 0-based labels)
        local_to_global: dict mapping local label → original 17-class label
    """
    paths, orig_labels = [], []

    for class_name, class_id in class_mapping.items():
        if GROUP_MAP[class_name] != group_id:
            continue   # skip classes not in this group
        class_dir = os.path.join(TRAIN_DIR, class_name)
        if not os.path.isdir(class_dir): continue
        for fname in os.listdir(class_dir):
            if fname.lower().endswith((".jpg", ".jpeg", ".png")):
                paths.append(os.path.join(class_dir, fname))
                orig_labels.append(class_id)   # store original ID

    # Remap original IDs to local 0-based IDs
    # e.g. [0, 8, 16] → {0:0, 8:1, 16:2}
    unique_orig = sorted(set(orig_labels))
    global_to_local = {orig: local for local, orig in enumerate(unique_orig)}
    local_to_global = {local: orig  for local, orig in enumerate(unique_orig)}
    local_labels    = [global_to_local[l] for l in orig_labels]

    tr_p, v_p, tr_l, v_l = train_test_split(
        paths, local_labels, test_size=0.2, random_state=SEED, stratify=local_labels
    )
    return tr_p, v_p, tr_l, v_l, local_to_global


# Preview what each Stage 2 model will see
print("Stage 2 datasets preview:")
for g in range(NUM_GROUPS):
    tr_p, v_p, tr_l, v_l, l2g = build_stage2_data(g)
    class_labels = [class_names[l2g[i]] for i in sorted(l2g)]
    print(f"  Group {g} ({GROUP_NAMES[g]}): {len(tr_p)} train | {len(v_p)} val | classes: {class_labels}")

In [ ]:
import gc, sys

stage2_models = {}

for group_id in range(NUM_GROUPS):
    print(f"\nPreparing Stage 2 — Group {group_id}: {GROUP_NAMES[group_id]}")
    try:
        tr_p, v_p, tr_l, v_l, local_to_global = build_stage2_data(group_id)
        num_local_classes = len(local_to_global)
        if num_local_classes == 1:
            print(f"  ⏭️ Skipping Group {group_id} — only 1 class")
            backbone_g, _ = load_eva_backbone()
            model_g = EVAClassifier(backbone_g, EMBED_DIM, 1).to(device)
            stage2_models[group_id] = (model_g.cpu(), local_to_global)
            continue

        counts  = np.bincount(tr_l)
        weights = [1.0 / counts[l] for l in tr_l]
        sampler = WeightedRandomSampler(weights, len(weights), replacement=True)

        tr_loader = DataLoader(
            SceneDataset(tr_p, tr_l, train_transform),
            batch_size=BATCH_SIZE, sampler=sampler,
            num_workers=4, pin_memory=True, persistent_workers=True, prefetch_factor=2,
            drop_last=True
        )
        v_loader = DataLoader(
            SceneDataset(v_p, v_l, val_transform),
            batch_size=BATCH_SIZE, shuffle=False,
            num_workers=4, pin_memory=True, persistent_workers=True, prefetch_factor=2
        )

        backbone_g, _ = load_eva_backbone()
        model_g = EVAClassifier(backbone_g, EMBED_DIM, num_local_classes).to(device)
        HARD_GROUPS = {0, 1}  # Modern/Minimal and Global/Maximalist need more capacity
        n_unfrozen = 8 if group_id in HARD_GROUPS else 4
        freeze_backbone_except_last_n_blocks(model_g, n_unfrozen_blocks=n_unfrozen)
        model_g = train_model(
            model        = model_g,
            train_loader = tr_loader,
            val_loader   = v_loader,
            num_epochs   = 12,
            backbone_lr  = 1e-5,
            head_lr      = 1e-4,
            label        = f'Stage2-Group{group_id} ({GROUP_NAMES[group_id]})'
        )

        torch.save(model_g.state_dict(), f"/kaggle/working/model_stage2_group{group_id}.pt")
        model_g = model_g.cpu()
        del backbone_g, tr_loader, v_loader
        stage2_models[group_id] = (model_g, local_to_global)
        print(f"Stage 2 Group {group_id} saved")

    except torch.cuda.OutOfMemoryError as e:
        print(f"  ❌ OOM on Group {group_id}: {e}")
        sys.last_traceback = None
        for var in ['model_g', 'backbone_g', 'tr_loader', 'v_loader', 'outputs', 'loss']:
            if var in dir():
                exec(f"del {var}")
        gc.collect()
        torch.cuda.empty_cache()
        try:
            stage2_models[group_id] = (EVAClassifier(load_eva_backbone()[0], EMBED_DIM, len(local_to_global)).cpu(), local_to_global)
        except Exception:
            pass
        print(f"  Skipping Group {group_id}, moving on...")
        continue

    gc.collect()
    torch.cuda.empty_cache()

print("\nAll Stage 2 models trained (or skipped if OOM)!")

In [ ]:
# ─────────────────────────────────────────────
# CELL 12: HIERARCHICAL INFERENCE FUNCTION  ← NEW
#
# This is where both stages are chained together.
# For a given image:
#   1. Stage 1 model predicts the group
#   2. The correct Stage 2 model predicts the fine class
#   3. We convert the local label back to the original 0-16 label
#
# We also keep TTA (test-time augmentation) like your original.
# ─────────────────────────────────────────────

base_transform_tta = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(INPUT_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(EVA_MEAN, EVA_STD)
])

tta_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(INPUT_SIZE),
    transforms.RandomHorizontalFlip(p=1.0),
    transforms.RandomRotation(degrees=10),
    transforms.ToTensor(),
    transforms.Normalize(EVA_MEAN, EVA_STD)
])


def get_probs_with_tta(model, image, n_aug=3):
    """Run model on image with TTA, return averaged softmax probabilities."""
    probs = torch.softmax(model(base_transform_tta(image).unsqueeze(0).to(device)), dim=1)
    for _ in range(n_aug):
        probs += torch.softmax(model(tta_transform(image).unsqueeze(0).to(device)), dim=1)
    return probs / (n_aug + 1)


def predict_hierarchical(img_path, n_aug=3):
    """
    Full hierarchical prediction for one image.
    Returns: original 0-16 class label
    """
    image = Image.open(img_path).convert('RGB')

    # ── Stage 1: predict the group ──────────────
    model_stage1.eval()
    with torch.no_grad():
        group_probs = get_probs_with_tta(model_stage1, image, n_aug)
        group_pred  = group_probs.argmax(1).item()   # 0, 1, 2, 3, or 4

    # ── Stage 2: predict class within that group ─
    model_g, local_to_global = stage2_models[group_pred]
    model_g = model_g.to(device)   # bring this group's model to GPU only when needed
    model_g.eval()
    with torch.no_grad():
        fine_probs  = get_probs_with_tta(model_g, image, n_aug)
        local_pred  = fine_probs.argmax(1).item()    # 0, 1, or 2 (local)
    model_g = model_g.cpu()        # free GPU memory again immediately after
    stage2_models[group_pred] = (model_g, local_to_global)

    # ── Convert local label back to original 0-16 label ─
    original_label = local_to_global[local_pred]
    return original_label


print("Hierarchical inference function ready")

In [ ]:
# CELL 12.5: END-TO-END VALIDATION EVALUATION
all_paths_eval, all_labels_eval = [], []
for class_name, class_id in class_mapping.items():
    class_dir = os.path.join(TRAIN_DIR, class_name)
    if not os.path.isdir(class_dir): continue
    for fname in os.listdir(class_dir):
        if fname.lower().endswith((".jpg", ".jpeg", ".png")):
            all_paths_eval.append(os.path.join(class_dir, fname))
            all_labels_eval.append(class_id)

_, val_paths_orig, _, val_labels_orig = train_test_split(
    all_paths_eval, all_labels_eval,
    test_size=0.2, random_state=SEED, stratify=all_labels_eval
)

correct = 0
model_stage1 = model_stage1.to(device)
for img_path, true_label in tqdm(zip(val_paths_orig, val_labels_orig)):
    pred = predict_hierarchical(img_path)
    if pred == true_label:
        correct += 1

print(f"Overall Hierarchical Val Accuracy: {correct/len(val_labels_orig)*100:.2f}%")

In [ ]:
# CELL: STAGE 1 — WHICH SPECIFIC CLASSES GET MISROUTED, AND WHERE

from collections import defaultdict

model_stage1 = model_stage1.to(device)
model_stage1.eval()

val_loader_orig = DataLoader(
    SceneDataset(val_paths_orig, val_labels_orig, val_transform),
    batch_size=BATCH_SIZE, shuffle=False,
    num_workers=4, pin_memory=True
)

class_to_group_pred = defaultdict(lambda: np.zeros(NUM_GROUPS, dtype=int))

with torch.no_grad():
    for images, labels in tqdm(val_loader_orig):
        images = images.to(device, non_blocking=True)
        with torch.cuda.amp.autocast():
            outputs = model_stage1(images)
        preds = outputs.argmax(1).cpu().numpy()
        for true_label, group_pred in zip(labels.numpy(), preds):
            class_to_group_pred[true_label][group_pred] += 1

print(f"{'Class':<20}{'TrueGroup':<28}" + "".join(f"{i:>6}" for i in range(NUM_GROUPS)))
for class_id in range(NUM_CLASSES):
    name = class_names[class_id]
    true_group = GROUP_MAP[name]
    row = class_to_group_pred[class_id]
    total = row.sum()
    acc = row[true_group]/total*100 if total>0 else 0
    row_str = "".join(f"{v:>6}" for v in row)
    print(f"{name:<20}{GROUP_NAMES[true_group]:<28}{row_str}   acc={acc:.1f}%")
model_stage1 = model_stage1.cpu()
torch.cuda.empty_cache()

In [ ]:
# ─────────────────────────────────────────────
# CELL 13: GENERATE SUBMISSION  ← slightly changed
#
# Same structure as your original submission cell,
# just calls predict_hierarchical() instead of tta_predict()
# ─────────────────────────────────────────────

submission_df  = pd.read_csv(SAMPLE_SUB_PATH)
label_col_name = submission_df.columns[1]
model_stage1 = model_stage1.to(device)

predictions = []
print(f"Processing {len(submission_df)} test images...")

for image_id in tqdm(submission_df.iloc[:, 0]):
    img_path = os.path.join(TEST_DIR, str(image_id))
    try:
        pred = predict_hierarchical(img_path, n_aug=3)
        predictions.append(pred)
    except (UnidentifiedImageError, FileNotFoundError):
        predictions.append(0)   # fallback for broken images

submission_df[label_col_name] = predictions
submission_df.to_csv("/kaggle/working/submission_hierarchical.csv", index=False)
print("✅ submission_hierarchical.csv saved!")
print(submission_df.head())

In [ ]:
import numpy as np, random
from PIL import Image

def sample_stats(paths, n=200):
    random.seed(0)
    sample = random.sample(paths, min(n, len(paths)))
    widths, heights, brightness = [], [], []
    for p in sample:
        try:
            img = Image.open(p).convert('RGB')
            widths.append(img.width); heights.append(img.height)
            arr = np.asarray(img.resize((64,64)), dtype=np.float32)
            brightness.append(arr.mean())
        except Exception:
            continue
    return np.array(widths), np.array(heights), np.array(brightness)

test_image_ids = pd.read_csv(SAMPLE_SUB_PATH).iloc[:, 0].tolist()
test_paths_sample = [os.path.join(TEST_DIR, str(i)) for i in test_image_ids]

tw, th, tb = sample_stats(train_paths, n=300)
ew, eh, eb = sample_stats(test_paths_sample, n=300)

print(f"TRAIN -> width:{tw.mean():.0f}±{tw.std():.0f}  height:{th.mean():.0f}±{th.std():.0f}  brightness:{tb.mean():.1f}±{tb.std():.1f}")
print(f"TEST  -> width:{ew.mean():.0f}±{ew.std():.0f}  height:{eh.mean():.0f}±{eh.std():.0f}  brightness:{eb.mean():.1f}±{eb.std():.1f}")

In [ ]:
print(submission_df[label_col_name].value_counts(normalize=True).sort_index() * 100)